# Day 2 | ILT 2: Lakeflow Connect + Storage Credentials & External Locations
### GlobalMart Data Engineering Bootcamp
---
**Session Time:** Day 2, Late Morning (90 min)
**Comes after:** ILT 1 — CDC Concepts (WAL & Logical Replication)
**Comes before:** HOL 1 — hands-on: create your own storage credential + external location, and stand up a Lakeflow Connect pipeline

---
### What We Cover Today
1. Recap — why watermark/JDBC polling doesn't scale, and what Lakeflow Connect replaces it with
2. How Lakeflow Connect works end to end (connection → pipeline → destination table), and its two connector modes (log-based vs query-based)
3. Worked example — GlobalMart's real `ecom_gbmart_conn` connection and the actual `orders_data_ingestion_cdc` pipeline for `orders` / `order_items`
4. Why hardcoded storage keys are a governance problem
5. The two Unity Catalog building blocks: **Storage Credential** and **External Location**
6. Worked example — GlobalMart's real `gbmart-ext-loc` location

> **Instructor note:** 90 minutes. ~15 min recap + motivation, ~30 min Lakeflow Connect live demo (UI walkthrough on `ecom_gbmart_conn` / `orders_data_ingestion_cdc` — point out it's query-based on `updated_at`, not WAL), ~15 min GlobalMart table assignment (Lakeflow Connect vs Autoloader), ~15 min storage-key problem + Unity Catalog building blocks, ~15 min live demo of `gbmart-ext-loc`. Do NOT have students create their own resources yet — that is HOL 1, immediately after this session.

## Section 1 — Recap: From Watermarks to a Managed Pipeline

In **ILT 1** we saw why a simple `WHERE updated_at > last_run` watermark falls apart:
- It misses **DELETEs** entirely — the row is just gone, nothing to filter on
- It only catches an UPDATE if the source table actually has an `updated_at` column
- At scale, re-scanning even the changed slice of a large table on a timer is wasteful

The gold-standard fix that captures every INSERT/UPDATE/DELETE with full fidelity is reading the **Write-Ahead Log (WAL)** directly through a **replication slot**. We did that by hand with raw JDBC in ILT 1 (and go deeper in HOL 2) so the mechanics are not a mystery.

**Lakeflow Connect can run either strategy as a managed pipeline — and it's important to know which one GlobalMart actually uses.** Lakeflow Connect supports two connector modes for a database source:

| Mode | How it detects changes | Catches DELETEs? |
|------|------------------------|-------------------|
| **Log-based (WAL / replication slot)** | Reads the database's write-ahead log directly | Yes — a DELETE is a WAL event like any other |
| **Query-based (cursor column)** | Runs `WHERE <cursor_col> > last_value` on a schedule, same idea as a watermark | **No** — same blind spot as manual watermarking |

> **GlobalMart's real, running pipeline (`orders_data_ingestion_cdc`) uses the query-based mode**, with `updated_at` as the cursor column — not WAL replication. This is a deliberate, honest trade-off you'll see in the worked example below: Databricks still manages the connection, scheduling, retries, and schema handling for you (that part of the "managed pipeline" value is real), but it does **not** magically give you DELETE-capture for free just because it's Lakeflow Connect. You only get that by explicitly configuring log-based replication, which requires more setup on the Postgres side (a replication slot has to exist and stay healthy).

```
Manual JDBC + WAL (ILT 1 / HOL 2)          Lakeflow Connect — query-based (GlobalMart's real pipeline)
──────────────────────────────────         ──────────────────────────────────────────────────────────
You open the JDBC connection                Databricks manages the connection
You create the replication slot             No replication slot — polls with a cursor column instead
You call pg_logical_slot_get_changes()       Databricks runs the cursor query on a schedule
You parse INSERT/UPDATE/DELETE text          Databricks applies INSERT/UPDATE only (no DELETE capture)
You write to Bronze yourself                 Lands as a governed, managed Unity Catalog table
Captures DELETEs, more code to maintain      Less code to maintain, same DELETE blind spot as a watermark
```

## Section 2 — How Lakeflow Connect Works End to End

```
Step 1: Connection
   A "Connection" object in Unity Catalog stores how to reach the source database
   (host, port, credentials). Created once, reused by any number of pipelines.

Step 2: Ingestion Pipeline
   You pick a Connection, pick which schema/tables to sync, pick a connector mode
   (log-based WAL, or query-based cursor column), and pick a destination catalog +
   schema in Unity Catalog. Databricks handles the rest.

Step 3: First Run — Full Snapshot
   The pipeline reads every row currently in the source table(s) and lands
   them as the initial state of a Unity Catalog managed table.

Step 4: Every Run After — Incremental Sync
   Databricks reads only what changed since the last run and applies it to
   the destination table — without re-reading the whole source table.
   HOW it detects "what changed" depends on the connector mode from Step 2:
     - Log-based:   reads new WAL events via the replication slot (catches DELETEs)
     - Query-based: re-queries WHERE cursor_column > last_seen_value (misses DELETEs)

Step 5: Destination
   A governed Unity Catalog table: <catalog>.<schema>.<table>
   Full lineage, access control, and audit — same as any other UC object.
```

### Why This Matters for `orders` and `order_items`

These two GlobalMart tables are **live, continuously changing** — order status moves from `pending` → `shipped` → `delivered`, line items get added or corrected. Re-reading the whole table on a timer (Autoloader-style full/incremental load) would either miss changes or waste enormous compute at GlobalMart's volume (~500K orders, ~2M order_items). Lakeflow Connect is the only tool in the GlobalMart stack built for this shape of source — even using the query-based mode, it still beats a hand-rolled watermark script because Databricks manages the connection, scheduling, retries, and schema evolution for you.

## Section 3 — Worked Example: GlobalMart's Real `orders_data_ingestion_cdc` Pipeline

This is the actual, currently-running Lakeflow Connect pipeline behind GlobalMart's Bronze `orders`/`order_items` tables — not a hypothetical. We use it as the live worked example — in HOL 1 you will create your **own**, named after yourself, pointing at your own Supabase project.

### Step 1 — The Connection (already created)

```
Databricks → Catalog → Connections → ecom_gbmart_conn

  Connection name : ecom_gbmart_conn
  Type            : PostgreSQL
  Host            : aws-0-ap-south-1.pooler.supabase.com
  Port            : 5432
```

This is a Unity Catalog Connection object — host/port/credentials are stored once, centrally, and referenced by name. No password ever appears in the pipeline definition itself.

### Step 2 — The Ingestion Pipeline (real spec, retrieved read-only from the workspace)

```
Pipeline name : orders_data_ingestion_cdc
Compute       : serverless          ← no cluster to size or manage
Connection    : ecom_gbmart_conn
Source type   : POSTGRESQL

  Source                                  Destination (Unity Catalog)
  ─────────────────────────────────       ──────────────────────────────
  postgres.globalmart.orders         →    gbmart.bronze.orders
  postgres.globalmart.order_items    →    gbmart.bronze.order_items

  Table configuration:
    primary_keys      : orderid           ← real source column, NO underscore
                         orderitemid       ← underscores (order_id) get added later, in Silver
    connector mode     : query_based
    cursor_columns     : updated_at       ← THIS is what makes it query-based, not WAL
```

> **Notice:** the source column is `orderid`, not `order_id`. Bronze lands data exactly as the source names it. The clean, underscored `order_id` you'll see from Day 5 onward is a Silver-layer standardization — don't expect it in Bronze.

### First Run Result (illustrative row counts)

```
Status     : Completed
Tables     : orders, order_items
orders     : Upserted ~500,000 rows   ← full snapshot on first run
order_items: Upserted ~2,000,000 rows ← full snapshot on first run
Catalog    : gbmart.bronze
```

### Every Run After — Only Rows Where `updated_at` Advanced

```
Someone updates order O-10021, which sets its updated_at to now()
  → Pipeline runs again
  → Upserted: 1     ← only rows whose updated_at moved, not all 500K
  → Bronze reflects the new status within one pipeline run

Someone instead DELETEs a row in Supabase entirely
  → Pipeline runs again
  → Nothing to upsert — a deleted row has no updated_at to trigger on
  → The row silently stays in gbmart.bronze.orders forever
  → This is the real, documented trade-off of query-based mode (see Section 1)
```

> **Verify (SQL you'll run for real in HOL 1):**
> ```sql
> SELECT COUNT(*) FROM gbmart.bronze.orders;
> DESCRIBE EXTENDED gbmart.bronze.orders;
> ```

## Section 4 — GlobalMart's Full Ingestion Tool Assignment

Not every table needs Lakeflow Connect. Match the tool to the source's shape:

| Table | Source | Tool | Strategy | Why |
|-------|--------|------|----------|-----|
| `orders` | Postgres (Supabase) | **Lakeflow Connect** | Query-based CDC (cursor: `updated_at`) | Continuously updated — INSERT + UPDATE tracked; DELETEs are not (see Section 3) |
| `order_items` | Postgres (Supabase) | **Lakeflow Connect** | Query-based CDC (cursor: `updated_at`) | Line items change with order status |
| `customers` | ADLS file drop | Autoloader | Incremental | New customers arrive daily — append + watermark |
| `products` | ADLS file drop | Autoloader | Full load | Catalog refreshed periodically — overwrite is safe |
| `addresses` | ADLS file drop | Autoloader | Incremental | Grows over time — append new rows |
| `payments` | ADLS file drop | Autoloader | Incremental | New payments arrive continuously |
| `payment_methods` | ADLS file drop | Autoloader | Full load | Small reference table — 4 rows |
| `shipping_tier` | ADLS file drop | Autoloader | Full load | Small reference table |
| `suppliers` | ADLS file drop | Autoloader | Full load | Small, infrequently updated |
| `returns` | ADLS file drop | Autoloader | Incremental | New returns arrive over time |

### Decision Rule

```
Is the source a live database with INSERT / UPDATE / DELETE happening continuously?
    → Lakeflow Connect
    (choose log-based/WAL if you must capture DELETEs; query-based/cursor if you don't)

Is the source a file that lands in ADLS and, once landed, never changes?
    Does the table grow over time (new rows accumulate)?
        → Autoloader + Incremental Load (append + watermark)
    Is the table small and fully refreshed each time?
        → Autoloader + Full Load (overwrite)
```

---
## Section 5 — The Other Half of Today: Why Hardcoded Storage Keys Are a Problem

Since Day 1, every notebook that reads ADLS has started with this pattern:

```python
storage_account_name = "YOUR_STORAGE_ACCOUNT_NAME"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"   # ← 88-character secret

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
```

### Why This Doesn't Scale to a Real Team

| Problem | What It Means |
|---------|---------------|
| **Security risk** | Anyone who opens the notebook sees a key with full read/write access to the entire storage account |
| **Git exposure** | If committed to GitHub, the key stays in git history even after being deleted from the file |
| **Key rotation** | When the storage key is rotated (security policy), every notebook that hardcodes it breaks at once |
| **No audit trail** | Azure only sees "the key was used" — not which user, from which notebook |
| **All-or-nothing access** | Everyone with the key has full read/write. You cannot give one team read-only |

```
CURRENT FLOW (key-based):
  Notebook → spark.conf.set(key) → Spark driver → ADLS
                    ↑
              secret lives in plain text in the notebook cell

WHAT WE WANT:
  Notebook → abfss://... → Unity Catalog checks permission → ADLS
                                        ↑
                              no key anywhere — auth handled centrally
```

## Section 6 — The Two Unity Catalog Building Blocks

### Building Block 1 — Storage Credential

```
A Storage Credential is Databricks' identity card for Azure.

It wraps an Azure Managed Identity (the Databricks Access Connector) —
an Azure resource granted "Storage Blob Data Contributor" on your ADLS
storage account.

When Databricks needs to read/write ADLS, it uses this managed identity
to authenticate. No password, no key, nothing that can leak.
```

**Analogy:** the Storage Credential is an employee badge — it proves "this is who Databricks is" to Azure.

### Building Block 2 — External Location

```
An External Location is a registered ADLS path + which Storage Credential
to use for it.

  External Location: gbmart-ext-loc
    URL:                abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/
    Storage Credential: ecomprojectscredentials
```

**Analogy:** the External Location is the door the badge unlocks — it maps a specific room (an ADLS path) to a specific badge (a credential).

### The Full Auth Chain

```
Notebook uses abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data/
       ↓
Unity Catalog: "This path is covered by External Location 'gbmart-ext-loc'"
       ↓
Storage Credential: "Use managed identity 'ecomprojectscredentials'"
       ↓
Azure RBAC: "That managed identity has Storage Blob Data Contributor on ecomadlsdata"
       ↓
ADLS Gen2: access granted — data returned

At no point does a key appear anywhere in the notebook.
```

## Section 7 — Worked Example: GlobalMart's Real External Location

Both objects already exist in this workspace, backing the real GlobalMart Bronze pipeline. We use them as the live worked example — in HOL 1 you will create your **own** storage credential + external location, named after yourself, pointing at your **own** storage account from Day 1.

```
Databricks → Catalog icon → External Data → Storage Credentials
  ecomprojectscredentials          Azure Managed Identity     Active

Databricks → Catalog icon → External Data → External Locations
  gbmart-ext-loc
    URL                : abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/
    Storage credential  : ecomprojectscredentials
```

This is the exact location the Autoloader Bronze notebooks (Day 3 / Day 4) read from — the base path `.../ecom-gbmart-data@ecomadlsdata.../raw-data` plus a subfolder per source (`customers/`, `products/`, `payments/`, etc.).

### Before vs After — What Changes in Every Notebook

**BEFORE (key-based auth):**
```python
storage_account_name = "YOUR_STORAGE_ACCOUNT_NAME"
container_name       = "amazon-data"
storage_account_key  = "YOUR_STORAGE_ACCOUNT_KEY"   # ← secret in the notebook

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)
```

**AFTER (Unity Catalog external location):**
```python
# No setup cell needed — Unity Catalog handles authentication via the
# External Location. The path is just a string, not a secret.
base_path = "abfss://ecom-gbmart-data@ecomadlsdata.dfs.core.windows.net/raw-data"
```

### Verify in SQL

```sql
SHOW EXTERNAL LOCATIONS;
DESCRIBE EXTERNAL LOCATION `gbmart-ext-loc`;

-- Grant access to a teammate — no key sharing required:
GRANT READ FILES ON EXTERNAL LOCATION `gbmart-ext-loc` TO `teammate@company.com`;
```

> **In HOL 1** you will create your own Access Connector–backed Storage Credential and External Location against your own Day-1 storage account, run `dbutils.fs.ls()` against it with zero keys in the notebook, and then build a Lakeflow Connect pipeline of your own against your own Supabase project. The `gbmart-ext-loc` shown above stays the shared, real location the rest of the course's Bronze/Silver/Gold pipeline actually reads from — your personal one is for practicing the skill.

---
## Recap

| Topic | Key Takeaway |
|-------|--------------|
| Lakeflow Connect | Managed Databricks pipeline; can run log-based (WAL) or query-based (cursor column) — GlobalMart's real pipeline uses query-based |
| Connection | Stores how to reach the source DB — created once, reused by pipelines (`ecom_gbmart_conn`) |
| Ingestion Pipeline | Full snapshot on first run; INSERT/UPDATE tracked via `updated_at` cursor on every run after — DELETEs are NOT captured in query-based mode |
| GlobalMart split | `orders` / `order_items` → Lakeflow Connect (query-based). Everything else → Autoloader |
| Storage Credential | Databricks' managed-identity "badge" for talking to Azure — no key |
| External Location | Maps an ADLS path to the credential that's allowed to access it |
| Real example | `gbmart-ext-loc` → `ecomprojectscredentials` → `ecomadlsdata` storage account |
| Governance win | Grant/revoke per user, full audit log, no key rotation to chase across notebooks |

---

## What Comes Next

| Session | Topic |
|---------|-------|
| **HOL 1 (next)** | Hands-on — create your own Storage Credential + External Location, and your own Lakeflow Connect pipeline for `orders` / `order_items` |
| **HOL 2** | Hands-on — WAL/replication-slot mechanics directly via JDBC (the log-based mode Lakeflow Connect *could* use, but GlobalMart's real pipeline doesn't) |
| **ILT 3 / HOL 3** | Code Versioning — Databricks Repos + GitHub |

---

**INSTRUCTOR NOTE:**
Closing check:
1. *'What does a Storage Credential actually contain?'* (Nothing secret you can see — it wraps an Azure Managed Identity via the Access Connector.)
2. *'If GlobalMart rotates the ADLS storage key today, what breaks?'* (Nothing — Unity Catalog external locations don't use the account key at all.)
3. *'Why is `orders` on Lakeflow Connect and `products` on Autoloader?'* (`orders` changes continuously with INSERT/UPDATE in a live database; `products` arrives as files that don't change once landed.)
4. *'If someone hard-deletes a row in the source `orders` table, will it disappear from `gbmart.bronze.orders`?'* (No — the real pipeline is query-based on `updated_at`, which has no way to notice a row is gone. This is the same blind spot as a manual watermark. Only log-based/WAL replication would catch it.)